# Tagout — Harvest Data Exploration
Initial look at Idaho F&G harvest statistics after ingestion.

In [ ]:
import duckdb
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

DB_PATH = Path('../data/tagout.duckdb')
con = duckdb.connect(str(DB_PATH))

In [ ]:
# Overview
con.execute('SELECT * FROM harvest LIMIT 5').df()

In [ ]:
# Success rate by species over time
df = con.execute("""
    SELECT species, season_year, AVG(success_pct) as avg_success, SUM(kills) as total_kills
    FROM harvest
    GROUP BY species, season_year
    ORDER BY species, season_year
""").df()

fig, ax = plt.subplots(figsize=(14, 6))
for species, group in df.groupby('species'):
    ax.plot(group['season_year'], group['avg_success'], marker='o', label=species)
ax.set_title('Harvest Success Rate by Species (Idaho)')
ax.set_xlabel('Year')
ax.set_ylabel('Success %')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Top hunt units by success rate (deer)
con.execute("""
    SELECT hunt_unit, AVG(success_pct) as avg_success, SUM(kills) as total_kills, COUNT(*) as seasons
    FROM harvest
    WHERE species = 'Deer'
    GROUP BY hunt_unit
    HAVING seasons >= 5
    ORDER BY avg_success DESC
    LIMIT 20
""").df()